In [2]:
import numpy as np
import pandas as pd
import seaborn as sns

In [3]:
df = pd.read_csv('../data/feautred.csv')
np.random.seed(42)

In [4]:
df.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'loan_term_months', 'employees_count',
       'business_type', 'is_rev_line_of_loan', 'loan_amount_rub',
       'loan_status', 'is_franchise', 'industry_sector', 'business_age_months',
       'okved_risk_level', 'authorized_capital', 'revenue_last_year',
       'revenue_growth_rate', 'net_profit', 'has_tax_arrears',
       'has_blocked_accounts', 'bki_score', 'current_debt_load',
       'max_delinquency_days', 'active_credits_count', 'fssp_debt_amount',
       'active_lawsuits_count', 'loan_purpose', 'ownership_structure',
       'business_scale'],
      dtype='str')

In [5]:
df.head()

,Unnamed: 0.1,Unnamed: 0,loan_term_months,employees_count,business_type,is_rev_line_of_loan,loan_amount_rub,loan_status,is_franchise,industry_sector,...,has_blocked_accounts,bki_score,current_debt_load,max_delinquency_days,active_credits_count,fssp_debt_amount,active_lawsuits_count,loan_purpose,ownership_structure,business_scale
0,0,0,84,1.609438,2,0,15.408819,1,0,45,...,0,782,6.999399e+06,0,4,0.0,0,working_capital,2,1
1,1,1,60,1.098612,2,0,15.003354,1,0,72,...,0,800,2.626400e+06,0,1,5000.0,0,investment,2,1
2,2,2,180,2.079442,1,0,16.973957,1,0,62,...,0,724,5.968333e+07,0,2,0.0,0,working_capital,1,1
3,3,3,60,1.098612,1,0,14.869823,1,0,unknown,...,0,833,4.564599e+06,0,1,0.0,2,working_capital,2,1
4,4,4,240,2.708050,1,0,16.748197,1,0,unknown,...,1,672,2.465180e+07,0,0,0.0,0,working_capital,2,1


#### Масштабируем признаки

In [17]:
from sklearn.preprocessing import StandardScaler

numeric_cols = [
    "loan_term_months",
    "employees_count",
    "loan_amount_rub",
    "business_age_months",
    "authorized_capital",
    "revenue_last_year",
    "revenue_growth_rate",
    "net_profit",
    "bki_score",
    "current_debt_load",
    "max_delinquency_days",
    "active_credits_count",
    "fssp_debt_amount",
    "active_lawsuits_count",
]

X_numeric_source = df[numeric_cols].copy()

# Эти признаки ещё не логарифмированы.
log_cols = [
    "authorized_capital",
    "revenue_last_year",
    "current_debt_load",
    "fssp_debt_amount",
]

X_numeric_source[log_cols] = np.log1p(X_numeric_source[log_cols])

X_numeric_source["net_profit"] = (
    np.sign(X_numeric_source["net_profit"])
    * np.log1p(np.abs(X_numeric_source["net_profit"]))
)

scaler = StandardScaler()

X_numeric = pd.DataFrame(
    scaler.fit_transform(X_numeric_source),
    columns=numeric_cols,
    index=df.index,
)

#### Кодируем(BinaryEncoder) признаки

In [18]:
from category_encoders import BinaryEncoder

categorical_cols = [
    "business_type",
    "industry_sector",
    "loan_purpose",
    "ownership_structure",
    "business_scale",
    "okved_risk_level",
]

encoder = BinaryEncoder(
    cols=categorical_cols,
    handle_unknown="value",
    handle_missing="value",
    return_df=True,
)

X_encoded = encoder.fit_transform(
    df[categorical_cols].astype(str)
)

Объединяем признаки

In [19]:
binary_cols = [
    "is_rev_line_of_loan",
    "is_franchise",
    "has_tax_arrears",
    "has_blocked_accounts",
]

X_cluster = pd.concat(
    [
        X_numeric,
        X_encoded,
        df[binary_cols],
    ],
    axis=1,
).astype("float32")

print(X_cluster.shape)

(871007, 34)


#### обучаем MiniBatchKMeans

In [20]:
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score

# Одна и та же выборка только для оценки качества.
X_eval = X_cluster.sample(n=2000, random_state=42)

results = []
best_score = -np.inf

for k in [10, 15, 20]:
    model = MiniBatchKMeans(
        n_clusters=k,
        batch_size=8192,
        n_init=10,
        max_iter=200,
        random_state=42,
    )

    # Обучение на всех данных.
    model.fit(X_cluster)

    score = silhouette_score(
        X_eval,
        model.predict(X_eval),
    )

    results.append({
        "k": k,
        "silhouette": score,
    })

    # Сохраняем лучший обученный вариант в памяти.
    if score > best_score:
        best_score = score
        cluster_model = model

    print(f"k={k}: silhouette={score:.4f}")

cluster_results = pd.DataFrame(results).sort_values(
    "silhouette",
    ascending=False,
)

print("\nСравнение вариантов:")
print(cluster_results.to_string(index=False))

# Результат лучшей по silhouette модели для каждой строки.
df_clusters = df.copy()
df_clusters["cluster"] = cluster_model.predict(X_cluster)

print("\nВыбрано кластеров:", cluster_model.n_clusters)

k=10: silhouette=0.0893
k=15: silhouette=0.0851
k=20: silhouette=0.0776

Сравнение вариантов:
 k  silhouette
10    0.089271
15    0.085108
20    0.077600

Выбрано кластеров: 10


In [30]:
# Размеры кластеров.
cluster_sizes = (
    df_clusters.groupby("cluster")
    .size()
    .rename("rows")
    .to_frame()
)

cluster_sizes["share_pct"] = (
    cluster_sizes["rows"] / len(df_clusters) * 100
).round(2)

print("Размеры кластеров:")
print(cluster_sizes.to_string())

# Возвращаем сумму и число сотрудников в исходную шкалу.
profile_data = df_clusters[
    ["cluster"] + numeric_cols
].copy()

profile_data["loan_amount_rub"] = np.exp(
    profile_data["loan_amount_rub"]
)

profile_data["employees_count"] = np.rint(
    np.exp(profile_data["employees_count"])
)

print("\nМедианы числовых признаков:")
print(
    profile_data.groupby("cluster")[numeric_cols]
    .median()
    .round(2)
)

print("\nСамые частые категории:")
print(
    df_clusters.groupby("cluster")[categorical_cols]
    .agg(lambda x: x.value_counts().idxmax())
    .to_string()
)

print("\nДоля единиц в бинарных признаках:")
print(
    df_clusters.groupby("cluster")[binary_cols]
    .mean()
    .round(3)
    .to_string()
)

Размеры кластеров:
           rows  share_pct
cluster                   
0         82692       9.49
1        120742      13.86
2         52458       6.02
3         96833      11.12
4         87546      10.05
5        104758      12.03
6         52206       5.99
7         98651      11.33
8        125076      14.36
9         50045       5.75

Медианы числовых признаков:
         loan_term_months  employees_count  loan_amount_rub  \
cluster                                                       
0                    84.0              4.0        2050001.0   
1                   240.0             11.0       30176001.0   
2                    48.0              3.0        1951847.0   
3                   120.0              5.0       13366001.0   
4                    84.0             24.0       19270001.0   
5                    84.0              4.0        4100001.0   
6                    44.0              4.0        4920001.0   
7                    84.0              3.0        3111819.0  

#### Поиск границ кластеров

In [31]:
cols = ['loan_amount_rub', 'loan_term_months', 'employees_count']
bounds = profile_data.groupby('cluster')[cols].quantile([0.25, 0.5, 0.75]).unstack()

print(bounds.round(0).to_string())

        loan_amount_rub                         loan_term_months               employees_count            
                   0.25        0.50        0.75             0.25   0.50   0.75            0.25  0.50  0.75
cluster                                                                                                   
0             1230001.0   2050001.0   3239001.0             60.0   84.0   84.0             2.0   4.0   6.0
1            18450001.0  30176001.0  48100381.0            240.0  240.0  300.0             6.0  11.0  22.0
2              820001.0   1951847.0   3066801.0             29.0   48.0   64.0             2.0   3.0   5.0
3             9020001.0  13366001.0  24600001.0             84.0  120.0  180.0             3.0   5.0   9.0
4            11070001.0  19270001.0  32800001.0             60.0   84.0  120.0            16.0  24.0  40.0
5             2443601.0   4100001.0   5740001.0             60.0   84.0   84.0             3.0   4.0   7.0
6             3378401.0   4920001.0  

По полученным данным в соответсвтии с таблицей ниже можно выделить 7 видов кредита

Продукты получены по кластеризации заёмщиков (MiniBatchKMeans, k=10). Суммы и сроки взяты из диапазона 25-75% внутри кластеров и округлены.

| Продукт | Группа | Кластеры | Сумма, млн ₽ | Срок, мес. | Цель | Для кого | Ставка, % |
|---|---|---|---|---|---|---|---|
| 1. Быстрый оборот | A | 2, 6 | 0.8-8 | 24-60 | оборотные средства | микро и малый бизнес, нужны короткие деньги | 18-21 |
| 2. Малый бизнес Стандарт | B | 0, 5, 7 | 1-6 | 60-84 | оборотные средства, рефинансирование | малый бизнес и ИП, средний чек | 16-19 |
| 3. Развитие | C | 8 | 7-15 | 84-120 | расширение, оборотные средства | растущий малый бизнес | 15-18 |
| 4. Инвестиционный | D | 3 | 9-25 | 84-180 | покупка оборудования | бизнес с инвестиционными планами | 14-17 |
| 5. Оборотный Плюс | E | 9 | 9-27 | 36-96 | оборотные средства, исполнение контрактов | растущий бизнес, госзакупки | 15-18 |
| 6. Средний бизнес | F | 4 | 11-33 | 60-120 | оборотные средства, инвестиции | 16-40 сотрудников | 13-16 |
| 7. Долгосрочный инвестиционный | G | 1 | 18-48 | 240-300 | недвижимость, крупные активы | крупные вложения на долгий срок | 12-15 |

## Примечания

- Чем больше сумма и дольше срок, тем ниже ставка: такие заёмщики в данных надёжнее.
- Продукты 1 и 2 близки по сумме, различаются сроком (короткий против среднего) и целью.
- Срок 240-300 месяцев для продукта 7 лучше подавать как кредит под залог недвижимости.

#### Сохраняем модель и encoder

In [36]:
import joblib
from pathlib import Path

artifacts_dir = Path("../artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)

cluster_to_product = {
    2: 'fast_working_capital', 6: 'fast_working_capital',
    0: 'small_business_standard', 5: 'small_business_standard', 7: 'small_business_standard',
    8: 'development',
    3: 'investment',
    9: 'working_capital_plus',
    4: 'medium_business',
    1: 'long_term_investment',
}

df_clusters['loan_product'] = df_clusters['cluster'].map(cluster_to_product)
df_clusters.drop(columns=['cluster']).to_csv('../data/featured_products.csv', index=False)

bundle = {
    "model": cluster_model,
    "scaler": scaler,
    "encoder": encoder,
    "numeric_cols": numeric_cols,
    "categorical_cols": categorical_cols,
    "binary_cols": binary_cols,
    "feature_columns": X_cluster.columns.tolist(),
    "log1p_cols": [
        "authorized_capital",
        "revenue_last_year",
        "current_debt_load",
        "fssp_debt_amount",
    ],
    "signed_log1p_cols": ["net_profit"],
    "already_log1p_cols": [
        "employees_count",
        "loan_amount_rub",
    ],
}

joblib.dump(
    bundle,
    artifacts_dir / "clustering_bundle.joblib",
    compress=3,
)

print("Модель и трансформеры сохранены")

Модель и трансформеры сохранены
